In [10]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold 

In [11]:
train = pd.read_csv('datadahdibersihinpasya/train_optimal.csv')
test = pd.read_csv('datadahdibersihinpasya/test_newreduced.csv')

In [12]:
train.head()

,Emisi Savanna Api,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Ritel Makanan,Emisi Pupuk Kandang Di Padang Rumput,Emisi Kebakaran Di Tanah Organik,Emisi Total,Peningkatan Suhu Rata - Rata ° C,Tahun,Negara_Encoded
0,2.816827,0.693147,2.625211,4.175130,-13.367805,0.0,4.715320,7.373080,0.0,7.902014,0.536167,1990,0
1,2.816827,0.693147,2.618277,4.145443,-13.367805,0.0,4.776421,7.414112,0.0,7.947195,0.020667,1991,0
2,2.816827,0.693147,2.618277,4.011870,-13.367805,0.0,4.853373,7.411862,0.0,7.958598,-0.259583,1992,0
3,2.816827,0.693147,2.618277,4.030602,-13.367805,0.0,4.424375,7.405472,0.0,7.962843,0.101917,1993,0
4,2.816827,0.693147,2.618277,4.023931,-13.367805,0.0,4.526135,7.433287,0.0,8.007875,0.372250,1994,0


In [13]:
test.head()

,Emisi Savanna Api,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Ritel Makanan,Emisi Pupuk Kandang Di Padang Rumput,Emisi Kebakaran Di Tanah Organik,Emisi Total,Tahun,Negara_Encoded
0,0.8454,0.0,81.852555,440.0315,-246.2191,0.0,370.6039,2719.1528,0.0,12652.876111,2015,0
1,1.6558,0.0,54.909681,340.8931,154.6574,0.0,425.9346,2692.9570,0.0,12988.384380,2016,0
2,0.4015,0.0,55.148427,345.7609,154.6574,0.0,477.3314,2680.8381,0.0,12786.218762,2017,0
3,0.2008,0.0,72.743150,407.6310,154.6574,0.0,534.8263,2716.9079,0.0,13054.982649,2018,0
4,7.1050,0.0,80.806938,489.7252,154.6574,0.0,590.6114,2557.4329,0.0,13354.360473,2019,0


In [14]:
train.shape

(3601, 13)

In [15]:
test.shape

(1362, 12)

In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Define target and feature columns
target_col = "Peningkatan Suhu Rata - Rata ° C"
feature_cols = ["Emisi Savanna Api", "Emisi Tanah Organik Yang Dikeringkan (Co2)",
                "Emisi Pembuatan Pestisida", "Emisi Transportasi Makanan", "Lahan Hutan",
                "Konversi Hutan Bersih", "Emisi Ritel Makanan", "Emisi Pupuk Kandang Di Padang Rumput",
                "Emisi Kebakaran Di Tanah Organik", "Emisi Total"]

# Drop missing values
train.dropna(inplace=True)
test.dropna(inplace=True)

# Scale features
scaler = MinMaxScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# Function to create sequences
def create_sequences(data, target_col, seq_length, is_train=True):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data.iloc[i:(i + seq_length)][feature_cols].values
        xs.append(x)
        if is_train:
            ys.append(data.iloc[i + seq_length][target_col])
    if is_train:
        return np.array(xs), np.array(ys)
    return np.array(xs)

# Set sequence length
seq_length = 10

# Create sequences for training
X_train, y_train = create_sequences(train, target_col, seq_length, is_train=True)

# Create sequences for testing
X_test = create_sequences(test, target_col, seq_length, is_train=False)

# Reshape input for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], len(feature_cols)))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], len(feature_cols)))

# Build LSTM model with Dropout and ReduceLROnPlateau
model = Sequential([
    LSTM(100, activation='relu', return_sequences=True, input_shape=(seq_length, len(feature_cols))),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mae')

# Callbacks
early_stopping = EarlyStopping(monitor='loss', patience=50, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=20, min_lr=1e-6)

# Train the model
history = model.fit(X_train, y_train, epochs=900, batch_size=32, verbose=1, callbacks=[early_stopping, reduce_lr])

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Compute MAPE
mape_train6 = np.mean(np.abs((y_train - y_pred_train.flatten()) / y_train))
print(f"MAPE: {mape_train6:.5f}")

# Convert predictions to DataFrame
test_predictions6 = pd.DataFrame(y_pred_test, columns=[target_col])

# Ensure 1362 rows by adding NaN rows
missing_rows = 1362 - len(test_predictions6)
padding = pd.DataFrame(np.nan, index=range(missing_rows), columns=[target_col])
test_predictions6 = pd.concat([padding, test_predictions6], ignore_index=True)

# Fill NaN values with backward fill
test_predictions6.fillna(method='bfill', inplace=True)


GPU Available: []
Epoch 1/900


d:\Anaconda\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.4809 - learning_rate: 0.0010
Epoch 2/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3750 - learning_rate: 0.0010
Epoch 3/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3733 - learning_rate: 0.0010
Epoch 4/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3588 - learning_rate: 0.0010
Epoch 5/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3707 - learning_rate: 0.0010
Epoch 6/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3583 - learning_rate: 0.0010
Epoch 7/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3655 - learning_rate: 0.0010
Epoch 8/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3618 - learning_rate: 0.0010
Epoch 9/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3538 - learning_rate: 0.0010
Epoch 10/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3579 - learning_rate: 0.0010
Epoch 11/900
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3559 - learning_rate: 0.001

C:\Users\fadhl\AppData\Local\Temp\ipykernel_20948\2752758569.py:84: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

C:\Users\fadhl\AppData\Local\Temp\ipykernel_20948\2752758569.py:87: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [21]:
print(f"Training MAPE: {mape_train6:.2f}%")

Training MAPE: 0.29%


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=20, min_lr=1e-6)
model.fit(X_train, y_train, epochs=1000, batch_size=32, callbacks=[early_stopping, reduce_lr])

mape = np.mean(np.abs((y_train - y_pred_train.flatten()) / y_train))
print(f"MAPE: {mape:.5f}")

In [22]:
# Save predictions to CSV
test_predictions6.to_csv("test_predictions_new.csv", index=False)

print("Final shape of test predictions:", test_predictions6.shape)

Final shape of test predictions: (1362, 1)


In [23]:
import numpy as np

def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Hindari pembagian dengan nol
    non_zero_mask = y_true != 0
    if np.sum(non_zero_mask) == 0:
        return np.nan  # Jika semua nilai 0, return NaN
    
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
